# Particle Swarm Optimization for Eggholder and Schwefel Functions
## Homework 6 Spring 2026

This notebook implements Particle Swarm Optimization (PSO) with various configurations and applies it to two challenging optimization problems:
- **Eggholder Function** (2D)
- **Schwefel Function** (4D)

The goal is to compare 10 different PSO variants across 5 seeds each (50 experiments per function) and analyze their performance.

## 1. Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
import seaborn as sns
from typing import Callable, Tuple, List, Dict
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)  # For reproducibility of initial setup

# Setup plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Environment setup complete.")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Objective Functions

In [ ]:
def eggholder(x):
    x1, x2 = x[0], x[1]
    term1 = -(x2 + 47) * np.sin(np.sqrt(np.abs(x2 + x1/2 + 47)))
    term2 = -x1 * np.sin(np.sqrt(np.abs(x1 - (x2 + 47))))
    return term1 + term2

def schwefel(x):
    n = len(x)
    alpha = 418.982887
    total = sum(-x[i] * np.sin(np.sqrt(np.abs(x[i]))) for i in range(n))
    return total + alpha * n

print('Testing objective functions:')
print(f'Eggholder at (512, 404.2319): {eggholder(np.array([512, 404.2319])):.4f}')
print(f'Schwefel at (420.97, 420.97, 420.97, 420.97): {schwefel(np.array([420.97, 420.97, 420.97, 420.97])):.4f}')

## 3. PSO Class Implementation

In [ ]:
class PSO:
    def __init__(self, obj_func, n_dims, bounds=(-512, 512), n_particles=30, max_iter=500,
                 phi1=2.0, phi2=2.0, use_inertia=False, use_constriction=False,
                 topology='star', model='full', seed=None):
        if seed is not None:
            np.random.seed(seed)
        
        self.obj_func = obj_func
        self.n_dims = n_dims
        self.bounds = bounds
        self.n_particles = n_particles
        self.max_iter = max_iter
        self.topology = topology
        self.model = model
        self.use_inertia = use_inertia
        self.use_constriction = use_constriction
        
        if model == 'cognition':
            self.phi1 = phi1
            self.phi2 = 0.0
        elif model == 'social':
            self.phi1 = 0.0
            self.phi2 = phi2
        elif model == 'selfless':
            self.phi1 = 0.0
            self.phi2 = phi2
        else:
            self.phi1 = phi1
            self.phi2 = phi2
        
        if self.use_constriction:
            phi = self.phi1 + self.phi2
            sqrt_term = np.sqrt(phi * phi - 4 * phi)
            self.K = 2.0 / np.abs(2.0 - phi - sqrt_term)
        else:
            self.K = 1.0
        
        self.v_max = (bounds[1] - bounds[0]) / 2.0
        
        lb, ub = bounds
        self.x = np.random.uniform(lb, ub, (n_particles, n_dims))
        self.v = np.random.uniform(-5, 5, (n_particles, n_dims))
        
        self.f = np.array([obj_func(self.x[i]) for i in range(n_particles)])
        self.p_best = self.x.copy()
        self.f_best = self.f.copy()
        
        self.g_idx = np.argmin(self.f)
        self.g_best = self.x[self.g_idx].copy()
        self.f_g_best = self.f[self.g_idx]
        
        self.best_fitness_history = [self.f_g_best]
    
    def _get_social_best(self, i):
        if self.topology == 'star':
            if self.model == 'selfless':
                f_copy = self.f_best.copy()
                f_copy[i] = np.inf
                idx = np.argmin(f_copy)
                return self.p_best[idx]
            else:
                return self.g_best
        elif self.topology == 'ring':
            left = (i - 1) % self.n_particles
            right = (i + 1) % self.n_particles
            candidates_idx = [left, i, right]
            candidates_f = [self.f_best[j] for j in candidates_idx]
            local_best_idx = candidates_idx[np.argmin(candidates_f)]
            if self.model == 'selfless':
                candidates_idx_no_self = [left, right]
                candidates_f_no_self = [self.f_best[j] for j in candidates_idx_no_self]
                local_best_idx = candidates_idx_no_self[np.argmin(candidates_f_no_self)]
            return self.p_best[local_best_idx]
    
    def step(self):
        if self.use_inertia:
            w = 0.9 - (0.5 * self.current_iter / self.max_iter)
        else:
            w = 1.0
        
        lb, ub = self.bounds
        
        for i in range(self.n_particles):
            g_best = self._get_social_best(i)
            r1 = np.random.random(self.n_dims)
            cognitive = self.phi1 * r1 * (self.p_best[i] - self.x[i])
            r2 = np.random.random(self.n_dims)
            social = self.phi2 * r2 * (g_best - self.x[i])
            
            if self.use_inertia and self.use_constriction:
                self.v[i] = self.K * (w * self.v[i] + cognitive + social)
            elif self.use_inertia:
                self.v[i] = w * self.v[i] + cognitive + social
            elif self.use_constriction:
                self.v[i] = self.K * (self.v[i] + cognitive + social)
            else:
                self.v[i] = self.v[i] + cognitive + social
            
            self.v[i] = np.clip(self.v[i], -self.v_max, self.v_max)
            self.x[i] = self.x[i] + self.v[i]
            
            for d in range(self.n_dims):
                if self.x[i, d] < lb:
                    self.x[i, d] = lb
                    self.v[i, d] = 0
                elif self.x[i, d] > ub:
                    self.x[i, d] = ub
                    self.v[i, d] = 0
            
            self.f[i] = self.obj_func(self.x[i])
            
            if self.f[i] < self.f_best[i]:
                self.f_best[i] = self.f[i]
                self.p_best[i] = self.x[i].copy()
        
        self.g_idx = np.argmin(self.f_best)
        if self.f_best[self.g_idx] < self.f_g_best:
            self.f_g_best = self.f_best[self.g_idx]
            self.g_best = self.p_best[self.g_idx].copy()
        
        self.best_fitness_history.append(self.f_g_best)
    
    def optimize(self):
        for iteration in range(self.max_iter):
            self.current_iter = iteration
            self.step()
        return self.g_best, self.f_g_best, self.best_fitness_history

print('PSO class defined successfully.')

## 4. Experiment Configuration

In [ ]:
SEEDS = [1, 2, 3, 4, 5]

VARIANTS = {
    'Variant 1: Full, Star, Base': {'phi1': 2.0, 'phi2': 2.0, 'use_inertia': False, 'use_constriction': False, 'topology': 'star', 'model': 'full'},
    'Variant 2: Cognition Only': {'phi1': 2.0, 'phi2': 2.0, 'use_inertia': False, 'use_constriction': False, 'topology': 'star', 'model': 'cognition'},
    'Variant 3: Social Only': {'phi1': 2.0, 'phi2': 2.0, 'use_inertia': False, 'use_constriction': False, 'topology': 'star', 'model': 'social'},
    'Variant 4: Selfless': {'phi1': 2.0, 'phi2': 2.0, 'use_inertia': False, 'use_constriction': False, 'topology': 'star', 'model': 'selfless'},
    'Variant 5: Ring Topology': {'phi1': 2.0, 'phi2': 2.0, 'use_inertia': False, 'use_constriction': False, 'topology': 'ring', 'model': 'full'},
    'Variant 6: Inertia': {'phi1': 2.0, 'phi2': 2.0, 'use_inertia': True, 'use_constriction': False, 'topology': 'star', 'model': 'full'},
    'Variant 7: Constriction': {'phi1': 2.05, 'phi2': 2.05, 'use_inertia': False, 'use_constriction': True, 'topology': 'star', 'model': 'full'},
    'Variant 8: Inertia+Constriction': {'phi1': 2.05, 'phi2': 2.05, 'use_inertia': True, 'use_constriction': True, 'topology': 'star', 'model': 'full'},
    'Variant 9a: More Particles': {'phi1': 2.05, 'phi2': 2.05, 'use_inertia': True, 'use_constriction': True, 'topology': 'star', 'model': 'full', 'n_particles': 60},
    'Variant 9b: Fewer Particles': {'phi1': 2.05, 'phi2': 2.05, 'use_inertia': True, 'use_constriction': True, 'topology': 'star', 'model': 'full', 'n_particles': 15}
}

print(f'Total variants: {len(VARIANTS)}')
print(f'Seeds per variant: {len(SEEDS)}')
print(f'Total experiments per function: {len(VARIANTS) * len(SEEDS)}')

## 5. Eggholder Function Experiments

In [ ]:
eggholder_results = {}
eggholder_convergence = {}

print('Running Eggholder Function Experiments (2D)')
for variant_idx, (variant_name, params) in enumerate(VARIANTS.items(), 1):
    print(f'{variant_idx}. {variant_name}')
    variant_results = []
    variant_convergence = []
    
    for seed in SEEDS:
        n_particles = params.get('n_particles', 30)
        pso = PSO(obj_func=eggholder, n_dims=2, bounds=(-512, 512), n_particles=n_particles,
                  max_iter=500, phi1=params['phi1'], phi2=params['phi2'],
                  use_inertia=params['use_inertia'], use_constriction=params['use_constriction'],
                  topology=params['topology'], model=params['model'], seed=seed)
        best_pos, best_fit, history = pso.optimize()
        variant_results.append({'seed': seed, 'best_fitness': best_fit, 'best_position': best_pos, 'n_particles': n_particles})
        variant_convergence.append(history)
        print(f'  Seed {seed}: {best_fit:.4f}')
    
    eggholder_results[variant_name] = variant_results
    eggholder_convergence[variant_name] = variant_convergence

print('Eggholder experiments complete!')

## 6. Eggholder Results Summary

In [ ]:
eggholder_summary_data = []
for variant_name, results in eggholder_results.items():
    fitnesses = [r['best_fitness'] for r in results]
    n_particles = results[0]['n_particles']
    eggholder_summary_data.append({
        'Variant': variant_name.split(':')[0],
        'Mean': np.mean(fitnesses),
        'Std': np.std(fitnesses),
        'Best': np.min(fitnesses),
        'Worst': np.max(fitnesses),
        'Particles': n_particles
    })

eggholder_summary_df = pd.DataFrame(eggholder_summary_data)
print('EGGHOLDER FUNCTION - Results Summary')
print(eggholder_summary_df.to_string(index=False))

## 7. Schwefel Function Experiments

In [ ]:
schwefel_results = {}
schwefel_convergence = {}

print('Running Schwefel Function Experiments (4D)')
for variant_idx, (variant_name, params) in enumerate(VARIANTS.items(), 1):
    print(f'{variant_idx}. {variant_name}')
    variant_results = []
    variant_convergence = []
    
    for seed in SEEDS:
        n_particles_base = params.get('n_particles', 50)
        if n_particles_base == 60:
            n_particles = 100
        elif n_particles_base == 15:
            n_particles = 25
        else:
            n_particles = 50
        
        pso = PSO(obj_func=schwefel, n_dims=4, bounds=(-512, 512), n_particles=n_particles,
                  max_iter=500, phi1=params['phi1'], phi2=params['phi2'],
                  use_inertia=params['use_inertia'], use_constriction=params['use_constriction'],
                  topology=params['topology'], model=params['model'], seed=seed)
        best_pos, best_fit, history = pso.optimize()
        variant_results.append({'seed': seed, 'best_fitness': best_fit, 'best_position': best_pos, 'n_particles': n_particles})
        variant_convergence.append(history)
        print(f'  Seed {seed}: {best_fit:.4f}')
    
    schwefel_results[variant_name] = variant_results
    schwefel_convergence[variant_name] = variant_convergence

print('Schwefel experiments complete!')

## 8. Schwefel Results Summary

In [ ]:
schwefel_summary_data = []
for variant_name, results in schwefel_results.items():
    fitnesses = [r['best_fitness'] for r in results]
    n_particles = results[0]['n_particles']
    schwefel_summary_data.append({
        'Variant': variant_name.split(':')[0],
        'Mean': np.mean(fitnesses),
        'Std': np.std(fitnesses),
        'Best': np.min(fitnesses),
        'Worst': np.max(fitnesses),
        'Particles': n_particles
    })

schwefel_summary_df = pd.DataFrame(schwefel_summary_data)
print('SCHWEFEL FUNCTION - Results Summary')
print(schwefel_summary_df.to_string(index=False))

## 9. Convergence Analysis

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(16, 20))
axes = axes.flatten()

for idx, (variant_name, convergence_histories) in enumerate(eggholder_convergence.items()):
    ax = axes[idx]
    for seed_idx, history in enumerate(convergence_histories):
        ax.plot(history, label=f'Seed {SEEDS[seed_idx]}', linewidth=2, alpha=0.7)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Best Fitness')
    ax.set_title(variant_name.split(':')[0])
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Eggholder Function - Convergence Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Eggholder convergence plots displayed.')

## 10. Final Summary

In [ ]:
print('='*80)
print('PARTICLE SWARM OPTIMIZATION - FINAL SUMMARY')
print('='*80)
print(f'\nTotal variants tested: {len(VARIANTS)}')
print(f'Seeds per variant: {len(SEEDS)}')
print(f'Total experiments per function: {len(VARIANTS) * len(SEEDS)}')
print(f'Total experiments overall: {2 * len(VARIANTS) * len(SEEDS)}')
print(f'\nEggholder global optimum: f(512, 404.2319) = -959.6407')
print(f'Schwefel global optimum: f(420.97, ...) = 0.0000')
print('\nBest performing variants:')

egg_means = [np.mean([r['best_fitness'] for r in results]) for results in eggholder_results.values()]
best_idx = np.argmin(egg_means)
best_variant = list(eggholder_results.keys())[best_idx]
print(f'  Eggholder: {best_variant} (Mean: {egg_means[best_idx]:.4f})')

sch_means = [np.mean([r['best_fitness'] for r in results]) for results in schwefel_results.values()]
best_idx = np.argmin(sch_means)
best_variant = list(schwefel_results.keys())[best_idx]
print(f'  Schwefel: {best_variant} (Mean: {sch_means[best_idx]:.4f})')
print('\nExperiments complete!')